# NAME TAG Backend Environment

현재 백엔드 실행 환경과 공개된 핵심 API를 IPYNB에서 재현한다.

검증 범위: 앱 import, `/health`, 회원가입, `/me`, Brand 생성/목록, BrandState 조회.

In [ ]:
import os
import sys
from pathlib import Path

workspace = Path.cwd()
if workspace.name == "notebooks":
    workspace = workspace.parent
backend = workspace / "backend"
os.chdir(backend)
sys.path.insert(0, str(backend))

print({"workspace": str(workspace), "backend": str(backend)})

In [ ]:
from pathlib import Path
import os
import sys

workspace = Path.cwd()
for candidate in (workspace, *workspace.parents):
    if (candidate / "backend" / "app" / "main.py").exists():
        workspace = candidate
        break
backend = workspace / "backend"
os.chdir(backend)
sys.path.insert(0, str(backend))

from app.main import app  # type: ignore[import-not-found]

routes = []
for route in app.routes:
    if hasattr(route, "path"):
        routes.append(route.path)
    original_router = getattr(route, "original_router", None)
    if original_router is not None:
        routes.extend(child.path for child in original_router.routes)
routes = sorted(set(routes))
assert "/health" in routes
assert "/api/auth/signup" in routes
assert "/api/brands" in routes
print("app import: ok")
print("routes:", routes)

app import: ok
routes: ['/api/auth/login', '/api/auth/logout', '/api/auth/me', '/api/auth/signup', '/api/brands', '/api/brands/{brand_id}/history', '/api/brands/{brand_id}/proposals', '/api/brands/{brand_id}/proposals/{proposal_id}', '/api/brands/{brand_id}/proposals/{proposal_id}/apply', '/api/brands/{brand_id}/proposals/{proposal_id}/approve', '/api/brands/{brand_id}/proposals/{proposal_id}/reject', '/api/brands/{brand_id}/state', '/docs', '/docs/oauth2-redirect', '/health', '/openapi.json', '/redoc']


In [11]:
import subprocess

migration = subprocess.run(
    [sys.executable, "-m", "alembic", "upgrade", "head"],
    cwd=backend,
    capture_output=True,
    text=True,
)
print(migration.stdout)
print(migration.stderr)
assert migration.returncode == 0, migration.stderr


INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.



In [15]:
import os
import sys
from pathlib import Path

workspace = Path.cwd()
for candidate in (workspace, *workspace.parents):
    if (candidate / "backend" / "app" / "main.py").exists():
        workspace = candidate
        break
backend = workspace / "backend"
os.chdir(backend)
sys.path.insert(0, str(backend))

for module_name in list(sys.modules):
    if module_name == "app" or module_name.startswith("app."):
        del sys.modules[module_name]

from app.main import app  # type: ignore[import-not-found]
from fastapi.testclient import TestClient

client = TestClient(app)
print("app reloaded with database:", os.environ.get("DATABASE_URL", "sqlite:///./name_tag.db"))

app reloaded with database: sqlite:///./name_tag.db


In [6]:
for route in app.routes:
    original_router = getattr(route, "original_router", None)
    if original_router is not None:
        print(
            "included",
            type(original_router).__name__,
            [getattr(child, "path", None) for child in original_router.routes],
        )
    else:
        print(type(route).__name__, getattr(route, "path", None))

Route /openapi.json
Route /docs
Route /docs/oauth2-redirect
Route /redoc
included APIRouter ['/api/auth/signup', '/api/auth/login', '/api/auth/me', '/api/auth/logout']
included APIRouter ['/api/brands', '/api/brands']
included APIRouter ['/api/brands/{brand_id}/state']
included APIRouter ['/api/brands/{brand_id}/proposals', '/api/brands/{brand_id}/proposals', '/api/brands/{brand_id}/proposals/{proposal_id}', '/api/brands/{brand_id}/proposals/{proposal_id}/approve', '/api/brands/{brand_id}/proposals/{proposal_id}/apply', '/api/brands/{brand_id}/proposals/{proposal_id}/reject']
included APIRouter ['/api/brands/{brand_id}/history']
APIRoute /health


In [8]:
client = TestClient(app)
response = client.get("/health")
assert response.status_code == 200, response.text
assert response.json()["status"] == "ok"
print(response.json())

{'status': 'ok', 'service': 'NAME TAG', 'environment': 'development'}


In [16]:
from uuid import uuid4

client = TestClient(app)
email = f"notebook-{uuid4().hex[:12]}@example.com"
signup = client.post(
    "/api/auth/signup",
    json={"email": email, "password": "NotebookPassword123!"},
)
assert signup.status_code == 201, signup.text
assert "name_tag_session" in client.cookies

me = client.get("/api/auth/me")
assert me.status_code == 200, me.text

brand = client.post(
    "/api/brands",
    json={"name": "Notebook Brand", "description": "Core API check"},
)
assert brand.status_code == 201, brand.text
brand_id = brand.json()["id"]

brands = client.get("/api/brands")
assert brands.status_code == 200, brands.text
assert any(item["id"] == brand_id for item in brands.json())

state = client.get(f"/api/brands/{brand_id}/state")
assert state.status_code == 200, state.text
assert state.json()["brand_id"] == brand_id
print({"user": me.json(), "brand_id": brand_id, "state_version": state.json()["version"]})

{'user': {'user': {'id': '706a0b1e-197c-4dea-941f-f0fce642951f', 'email': 'notebook-4075d4593f5e@example.com'}}, 'brand_id': '2e33e389-d318-43a3-896e-38dd5a2d0dfe', 'state_version': 1}
